# Multivariate Missing Data

This example demonstrates missing data patterns with multivariate generators (VAR and Copula). Each series gets independent missing data while preserving the underlying correlation structure.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from synforecast.generators import CopulaGenerator, VARGenerator

## VAR Generator with Random Missing Data

Generate 3 correlated series from a VAR(1) model with 15% random missing data.

In [ ]:
var_params = {
    "min_length": 200,
    "max_length": 200,
    "freq": "D",
    "lag_order": 1,
    "missing_data": True,
    "missing_pattern": "random",
    "missing_rate": 0.15,
    "seed": 42,
}

var_gen = VARGenerator(engine="polars", **var_params)
df_var = var_gen.generate(n_series=3)

print(
    f"Generated {df_var['unique_id'].n_unique()} correlated series with VAR(1) model"
)
print(f"Total observations: {len(df_var)}")
df_var.head(30)

In [ ]:
print("Missing data statistics by series:")
for series_id in df_var["unique_id"].unique().sort():
    series_df = df_var.filter(pl.col("unique_id") == series_id)
    values = series_df["y"].to_numpy()
    nan_count = np.sum(np.isnan(values))
    nan_rate = nan_count / len(values)
    print(f"  {series_id}: {nan_count} missing ({nan_rate:.1%})")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in df_var["unique_id"].unique().to_list():
    series = df_var.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8, marker=".", markersize=2, linewidth=0.8)
ax.set_title("VAR Series with Random Missing Data (15%)")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Value")
ax.legend()
plt.tight_layout()
plt.show()

## Copula Generator with Block Missing Data

Generate correlated series using a Gaussian copula with week-long missing blocks, simulating synchronized outages.

In [ ]:
correlation_matrix = np.array([[1.0, 0.7, 0.3], [0.7, 1.0, 0.5], [0.3, 0.5, 1.0]])

copula_params = {
    "min_length": 200,
    "max_length": 200,
    "freq": "D",
    "copula_type": "gaussian",
    "correlation_matrix": correlation_matrix,
    "missing_data": True,
    "missing_pattern": "block",
    "missing_rate": 0.2,
    "missing_block_size": 7,
    "seed": 123,
}

copula_gen = CopulaGenerator(engine="polars", **copula_params)
df_copula = copula_gen.generate(n_series=3)

print(
    f"Generated {df_copula['unique_id'].n_unique()} correlated series with Gaussian copula"
)
print(f"Missing blocks of size: {copula_params['missing_block_size']} days")
df_copula.head(40)

In [ ]:
print("Block missing statistics by series:")
for series_id in df_copula["unique_id"].unique().sort():
    series_df = df_copula.filter(pl.col("unique_id") == series_id)
    values = series_df["y"].to_numpy()
    nan_count = np.sum(np.isnan(values))
    nan_rate = nan_count / len(values)

    max_consecutive = 0
    current_consecutive = 0
    for val in values:
        if np.isnan(val):
            current_consecutive += 1
            max_consecutive = max(max_consecutive, current_consecutive)
        else:
            current_consecutive = 0

    print(
        f"  {series_id}: {nan_count} missing ({nan_rate:.1%}), "
        f"max block: {max_consecutive} days"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in df_copula["unique_id"].unique().to_list():
    series = df_copula.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8, marker=".", markersize=2, linewidth=0.8)
ax.set_title("Copula Series with Block Missing Data (7-day blocks)")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Value")
ax.legend()
plt.tight_layout()
plt.show()

## VAR with Seasonal Missing Data (Weekend Gaps)

Simulate a year of VAR(2) data with weekly seasonal missing patterns, representing weekend reporting gaps.

In [ ]:
var_seasonal_params = {
    "min_length": 365,
    "max_length": 365,
    "freq": "D",
    "lag_order": 2,
    "missing_data": True,
    "missing_pattern": "seasonal",
    "missing_rate": 0.12,
    "missing_seasonal_period": 7,
    "seed": 456,
}

var_seasonal_gen = VARGenerator(engine="polars", **var_seasonal_params)
df_var_seasonal = var_seasonal_gen.generate(n_series=2)

print(
    f"Generated {df_var_seasonal['unique_id'].n_unique()} correlated series (1 year)"
)
print(
    f"Seasonal period: {var_seasonal_params['missing_seasonal_period']} days (weekly)"
)

print("\nSeasonal missing statistics:")
for series_id in df_var_seasonal["unique_id"].unique().sort():
    series_df = df_var_seasonal.filter(pl.col("unique_id") == series_id)
    values = series_df["y"].to_numpy()
    nan_count = np.sum(np.isnan(values))
    nan_rate = nan_count / len(values)
    print(f"  {series_id}: {nan_count} missing ({nan_rate:.1%})")

df_var_seasonal.head(30)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in df_var_seasonal["unique_id"].unique().to_list():
    series = df_var_seasonal.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8, marker=".", markersize=2, linewidth=0.8)
ax.set_title("VAR Series with Seasonal Missing Data (Weekend Gaps)")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Value")
ax.legend()
plt.tight_layout()
plt.show()

## Correlation Analysis with Missing Data

Compare the correlation between series with and without missing data to verify that the underlying structure is preserved.

In [ ]:
params_complete = {
    "min_length": 500,
    "max_length": 500,
    "freq": "D",
    "lag_order": 1,
    "missing_data": False,
    "seed": 789,
}

params_missing = {
    "min_length": 500,
    "max_length": 500,
    "freq": "D",
    "lag_order": 1,
    "missing_data": True,
    "missing_pattern": "random",
    "missing_rate": 0.25,
    "seed": 789,
}

gen_complete = VARGenerator(engine="polars", **params_complete)
df_complete = gen_complete.generate(n_series=2)

gen_missing = VARGenerator(engine="polars", **params_missing)
df_missing = gen_missing.generate(n_series=2)

# Calculate correlations
series_0_complete = df_complete.filter(pl.col("unique_id") == "0")[
    "y"
].to_numpy()
series_1_complete = df_complete.filter(pl.col("unique_id") == "1")[
    "y"
].to_numpy()
corr_complete = np.corrcoef(series_0_complete, series_1_complete)[0, 1]

series_0_missing = df_missing.filter(pl.col("unique_id") == "0")[
    "y"
].to_numpy()
series_1_missing = df_missing.filter(pl.col("unique_id") == "1")[
    "y"
].to_numpy()
mask = ~(np.isnan(series_0_missing) | np.isnan(series_1_missing))
corr_missing = np.corrcoef(series_0_missing[mask], series_1_missing[mask])[0, 1]

print(f"Correlation (complete data): {corr_complete:.3f}")
print(f"Correlation (25% missing):   {corr_missing:.3f}")
print(f"Correlation preserved:       {abs(corr_complete - corr_missing) < 0.1}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for uid in df_complete["unique_id"].unique().to_list():
    series = df_complete.filter(pl.col("unique_id") == uid)
    axes[0].plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
axes[0].set_title(f"Complete Data (corr={corr_complete:.3f})")
axes[0].set_xlabel("Timestamp")
axes[0].set_ylabel("Value")
axes[0].legend()

for uid in df_missing["unique_id"].unique().to_list():
    series = df_missing.filter(pl.col("unique_id") == uid)
    axes[1].plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8, marker=".", markersize=2, linewidth=0.8)
axes[1].set_title(f"25% Missing Data (corr={corr_missing:.3f})")
axes[1].set_xlabel("Timestamp")
axes[1].set_ylabel("Value")
axes[1].legend()

plt.tight_layout()
plt.show()